# 04 — SimpleSequentialChain

Neste notebook vamos aprender o conceito de **sequência de chains** utilizando `SimpleSequentialChain`.

A ideia é dividir uma tarefa maior em etapas menores. A saída de uma etapa é utilizada como entrada da próxima.

Neste exemplo teremos duas etapas:

```text
Conceito
   ↓
Chain 1 — explica o conceito
   ↓
Explicação
   ↓
Chain 2 — resume a explicação
   ↓
Resultado final
```

> **Observação:** `SimpleSequentialChain` e `LLMChain` são APIs legadas do LangChain. Estamos utilizando-as aqui porque fazem parte do conteúdo do curso e ajudam a entender o conceito de chains. Nas versões modernas do LangChain, esse tipo de fluxo costuma ser construído com LCEL.

## 1. Importando as bibliotecas

Vamos importar quatro componentes:

- `ChatOllama`: integração do LangChain com o Ollama.
- `PromptTemplate`: criação de prompts reutilizáveis.
- `LLMChain`: combina uma LLM com um prompt.
- `SimpleSequentialChain`: executa várias chains em sequência.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SimpleSequentialChain

## 2. Configurando o Qwen

Assim como nos notebooks anteriores, vamos utilizar o Qwen rodando localmente através do Ollama.

In [ ]:
llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)

## 3. Criando o primeiro PromptTemplate

A primeira etapa da nossa aplicação será explicar um conceito para um estudante iniciante.

O conceito será uma variável chamada `{conceito}`.

In [ ]:
prompt_explicacao = PromptTemplate.from_template(
    "Explique o conceito de {conceito} para um estudante iniciante de tecnologia."
)

## 4. Criando a primeira LLMChain

Agora combinamos o prompt com a nossa LLM utilizando `LLMChain`.

Podemos pensar nessa chain como:

```text
Entrada
   ↓
PromptTemplate
   ↓
Qwen
   ↓
Saída
```

In [ ]:
chain_explicacao = LLMChain(
    llm=llm,
    prompt=prompt_explicacao
)

## 5. Testando a primeira chain

Antes de criar a sequência, vamos testar a primeira chain sozinha.

In [ ]:
resultado_1 = chain_explicacao.invoke({
    "conceito": "RAG"
})

print(resultado_1)

A primeira chain recebe o conceito e gera uma explicação.

Essa explicação será justamente o conteúdo que queremos passar para a próxima etapa.

## 6. Criando o segundo PromptTemplate

Agora vamos criar a segunda etapa.

Ela receberá a explicação produzida pela primeira chain e deverá resumir esse conteúdo em uma única frase.

A variável dessa etapa será `{explicacao}`.

In [ ]:
prompt_resumo = PromptTemplate.from_template(
    "Resuma a explicação abaixo em uma frase simples:\n\n{explicacao}"
)

## 7. Criando a segunda LLMChain

In [ ]:
chain_resumo = LLMChain(
    llm=llm,
    prompt=prompt_resumo
)

## 8. Criando a SimpleSequentialChain

Agora temos duas chains independentes.

Vamos conectá-las utilizando `SimpleSequentialChain`.

In [ ]:
chain = SimpleSequentialChain(
    chains=[chain_explicacao, chain_resumo],
    verbose=True
)

### O que `chains=[...]` significa?

Estamos dizendo ao LangChain:

```text
Execute primeiro chain_explicacao.
Depois pegue a saída dela e envie para chain_resumo.
```

O fluxo fica:

```text
"RAG"
   ↓
chain_explicacao
   ↓
"RAG é uma técnica..."
   ↓
chain_resumo
   ↓
"RAG permite..."
```

## 9. Executando a sequência

Agora basta fornecer a entrada inicial.

Nesse exemplo, nossa entrada será `RAG`.

In [ ]:
resultado = chain.invoke("RAG")

## 10. Visualizando o resultado final

In [ ]:
print(resultado["output"])

## 11. O que aconteceu internamente?

Quando executamos:

```python
chain.invoke("RAG")
```

o LangChain executou aproximadamente este fluxo:

```text
"RAG"
    ↓
┌─────────────────────────────┐
│ Chain 1                     │
│                             │
│ PromptTemplate               │
│       ↓                     │
│ Qwen                        │
│       ↓                     │
│ Explicação sobre RAG        │
└──────────────┬──────────────┘
               ↓
┌─────────────────────────────┐
│ Chain 2                     │
│                             │
│ PromptTemplate               │
│       ↓                     │
│ Qwen                        │
│       ↓                     │
│ Resumo                      │
└──────────────┬──────────────┘
               ↓
         Resultado final
```

## 12. Código completo

Agora podemos juntar todo o exemplo:

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SimpleSequentialChain

llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)

prompt_explicacao = PromptTemplate.from_template(
    "Explique o conceito de {conceito} para um estudante iniciante de tecnologia."
)

chain_explicacao = LLMChain(
    llm=llm,
    prompt=prompt_explicacao
)

prompt_resumo = PromptTemplate.from_template(
    "Resuma a explicação abaixo em uma frase simples:\n\n{explicacao}"
)

chain_resumo = LLMChain(
    llm=llm,
    prompt=prompt_resumo
)

chain = SimpleSequentialChain(
    chains=[chain_explicacao, chain_resumo],
    verbose=True
)

resultado = chain.invoke("RAG")

print(resultado["output"])